In [2]:
import os 
import json 
import pandas as pd 

from dotenv import load_dotenv 
import google.generativeai as genai 
import streamlit as st 

c:\Users\Quang Minh\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('./data/horoscope_saved.csv')
df.head()

,sign,category,date,horoscope
0,aries,general,20200617,"There's a great day ahead of you, Aries. You'l..."
1,aries,general,20200618,People will understand and appreciate your des...
2,aries,general,20200619,You are very interested in technological break...
3,aries,general,20200620,Stress from overwork could have you feeling we...
4,aries,general,20200621,This is a good day to stand up for yourself an...


In [4]:
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
df

,sign,category,date,horoscope
0,aries,general,2020-06-17,"There's a great day ahead of you, Aries. You'l..."
1,aries,general,2020-06-18,People will understand and appreciate your des...
2,aries,general,2020-06-19,You are very interested in technological break...
3,aries,general,2020-06-20,Stress from overwork could have you feeling we...
4,aries,general,2020-06-21,This is a good day to stand up for yourself an...
...,...,...,...,...
21954,pisces,birthday,2021-06-12,Celebrate in style on your birthday in prepara...
21955,pisces,birthday,2021-06-13,Imagine your life as if it was exactly the way...
21956,pisces,birthday,2021-06-14,"Fun, playfulness, and humor are easy to manife..."
21957,pisces,birthday,2021-06-15,"Your birthday brings you a fresh start, as you..."


In [5]:
df.isna().sum()

sign         0
category     0
date         0
horoscope    0
dtype: int64

In [6]:
df.nunique()

sign            12
category         5
date           366
horoscope    12050
dtype: int64

In [7]:
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df.to_json(orient='records')
df

,sign,category,date,horoscope,day_of_week,month,year
0,aries,general,2020-06-17,"There's a great day ahead of you, Aries. You'l...",2,6,2020
1,aries,general,2020-06-18,People will understand and appreciate your des...,3,6,2020
2,aries,general,2020-06-19,You are very interested in technological break...,4,6,2020
3,aries,general,2020-06-20,Stress from overwork could have you feeling we...,5,6,2020
4,aries,general,2020-06-21,This is a good day to stand up for yourself an...,6,6,2020
...,...,...,...,...,...,...,...
21954,pisces,birthday,2021-06-12,Celebrate in style on your birthday in prepara...,5,6,2021
21955,pisces,birthday,2021-06-13,Imagine your life as if it was exactly the way...,6,6,2021
21956,pisces,birthday,2021-06-14,"Fun, playfulness, and humor are easy to manife...",0,6,2021
21957,pisces,birthday,2021-06-15,"Your birthday brings you a fresh start, as you...",1,6,2021


In [8]:
import requests
def get_zodiac_daily(sign):
    api_url = f'https://api.api-ninjas.com/v1/horoscope?zodiac={sign}'
    response = requests.get(api_url, headers={'X-Api-Key': st.secrets['api']['horoscope-api-key']})
    if response.status_code == requests.codes.ok:
        return response.json()
    else:
        return {"error": response.status_code, "message": response.text}
get_zodiac_daily('ARIES')

{'date': '2025-07-07',
 'sign': 'Aries',
 'horoscope': "Aries, you might feel that others aren't approaching matters with the seriousness you desire. Consider this a gentle nudge to ease up a bit yourself. Remember, life is a playful journey. In the vastness of the cosmos, we are just tiny particles drifting through space. Our existence is fleeting, just a moment in the grand timeline of the universe. So, try not to take yourself too seriously."}

In [5]:
zodiac_signs = [
    "aries", "taurus", "gemini", "cancer", "leo", "virgo",
    "libra", "scorpio", "sagittarius", "capricorn", "aquarius", "pisces"
]

def extract_zodiac_keywords(text):
    words = text.lower().split()
    return [sign for sign in zodiac_signs if sign in words]

# Example
user_input = "I'm a Leo sun, Aries moon, and Gemini rising"
print(extract_zodiac_keywords(user_input))
# Output: ['leo', 'pisces', 'gemini']


['aries', 'gemini', 'leo']


In [16]:
def get_zodiac_related_ideas_keyword_only(user_query, df, top_n=10):
    zodiac_signs = [
        "aries", "taurus", "gemini", "cancer", "leo", "virgo",
        "libra", "scorpio", "sagittarius", "capricorn", "aquarius", "pisces"
    ]
    
    # Match signs from the query
    found_signs = [sign for sign in zodiac_signs if sign in user_query.lower()]
    
    # Filter by sign if any found
    filtered_df = df[df["sign"].isin(found_signs)] if found_signs else df

    # Rank by how many words from the query appear in the horoscope
    user_words = set(user_query.lower().split())
    filtered_df["score"] = filtered_df["horoscope"].apply(
        lambda h: sum(word in h.lower() for word in user_words)
    )

    # Get top N results
    top_results = filtered_df.sort_values(by="score", ascending=False).head(top_n)
    return top_results[["sign", "date", "horoscope"]].reset_index(drop=True)

# Test the fallback function
fallback_results = get_zodiac_related_ideas_keyword_only("Tell me about aries in love", df)
fallback_results


C:\Users\Quang Minh\AppData\Local\Temp\ipykernel_7876\1519182136.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["score"] = filtered_df["horoscope"].apply(


,sign,date,horoscope
0,aries,2021-04-11,A discussion could take place in your home bet...
1,aries,2020-06-16,Instincts and intuition serve you well in publ...
2,aries,2020-07-10,"You are a natural nurturer, and no matter how ..."
3,aries,2020-08-17,Career and your public image may be important ...
4,aries,2021-04-14,You aren't one to put much stock in the meanin...
5,aries,2020-12-31,Issues regarding love and romance should be go...
6,aries,2021-03-27,The planetary alignment may bring about a situ...
7,aries,2020-11-30,This is a wonderful day for arranging a get-to...
8,aries,2020-11-12,"As a Aries, you have an innate gift for healin..."
9,aries,2020-11-15,"Be tactful when speaking today, Aries. With th..."


In [11]:
load_dotenv()
google_api_key = st.secrets['api']['api-key']
genai.configure(api_key=google_api_key)

In [ ]:
# model = genai.GenerativeModel("gemini-1.5-flash") #Gemini 1.5 Flash

In [ ]:
# prompt = "Bạn là ai?"
# res = model.generate_content(prompt)
# res

response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "T\u00f4i l\u00e0 m\u1ed9t m\u00f4 h\u00ecnh ng\u00f4n ng\u1eef l\u1edbn, \u0111\u01b0\u1ee3c hu\u1ea5n luy\u1ec7n b\u1edfi Google."
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "avg_logprobs": -0.1621614545583725
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 4,
        "candidates_token_count": 16,
        "total_token_count": 20
      },
      "model_version": "gemini-1.5-flash"
    }),
)

In [ ]:
# res.text

'Tôi là một mô hình ngôn ngữ lớn, được huấn luyện bởi Google.'

In [ ]:
zodiac_df = df

with open('config.json','r') as f :
    config = json.load(f)
    functions = config.get('functions')

model = genai.GenerativeModel("gemini-1.5-flash",
                            system_instruction=f"""
                                You are AstroBot, a personal astrologer. You are an expert in astrology and horoscopes and you will help customers to 
                                answer their questions about zodiac signs, horoscopes, and astrology-related topics.
                                Act like a human astrologer, you will answer the questions in a friendly and helpful manner.
                                Despite the questions being in English, you will always response in Vietnamese.
                                Provide the answer in a concise and clear manner, using simple language that is easy to understand.
                                Answer in one paragraph only, do not write too long.
                                Only one answer per question, do not write multiple answers.
                                Diversify your answers, do not repeat the same answer.
                                If questions are not related to astrology, zodiac signs, or horoscopes, you will say "Tôi xin lỗi, tôi không thể trả lời câu hỏi này vì nó không liên quan đến chiêm tinh học, cung hoàng đạo hoặc tử vi."
                                    """)

In [15]:
prompt = "cung hoàng đạo của tôi là Bạch Dương. Hãy cho tôi biết về vận mệnh của tôi hôm nay."
response = model.generate_content(prompt)
response.text

'Hôm nay, bạn, người thuộc cung Bạch Dương năng động và đầy nhiệt huyết, sẽ có một ngày tràn đầy năng lượng tích cực.  Sự tự tin của bạn sẽ giúp bạn vượt qua mọi thử thách một cách dễ dàng.  Tuy nhiên, hãy nhớ giữ bình tĩnh và tránh nóng vội trong các quyết định quan trọng nhé!  Hãy tận hưởng những khoảnh khắc vui vẻ và đừng quên dành thời gian cho bản thân.\n'